# 14 Event-Based Sentence Demo — Fast Real-Life Signing Logic

This notebook demonstrates the next testing logic for **Be My Ear / Be My Voice**.

Instead of waiting for repeated predictions, it uses:

```text
hand activity → short sign event → event-level Top-5 → sentence buffer → NLP sentence
```

This is better for real-life signing because a sign may last only **0.5–2 seconds**.

# 1. Imports

In [ ]:
from pathlib import Path
from collections import deque, defaultdict
import json
import time

import cv2
import mediapipe as mp
import mss
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# 2. Load project and NLP module

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.nlp.candidate_sentence_mode import candidate_sentence_mode

print("Project root:", PROJECT_ROOT)

# 3. Load WLASL2000 config and label map

In [ ]:
DEPLOY_DIR = PROJECT_ROOT / "app" / "models" / "ASL" / "WLASL2000"
CONFIG_FILE = DEPLOY_DIR / "wlasl2000_deployment_config.json"

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_PATH = Path(config["model_path"])
LABEL_MAP_PATH = Path(config["label_map_path"])

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    raw_label_map = json.load(f)

id_to_gloss = {int(k): v["gloss"] for k, v in raw_label_map.items()}
rules = config["confidence_rules"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQUENCE_LENGTH = int(config["sequence_length"])
BASE_FEATURE_SIZE = int(config["base_keypoint_shape"][1])
INPUT_SIZE = int(config["input_shape"][1])
NUM_CLASSES = int(config["num_classes"])

print("Model:", MODEL_PATH)
print("Device:", device)

# 4. Load normalisation stats

In [ ]:
def find_norm_stats_file():
    candidates = []

    if "norm_stats_path" in config:
        candidates.append(Path(config["norm_stats_path"]))

    model_dir = PROJECT_ROOT / "models" / "ASL" / "WLASL2000"
    selected_name = config.get("selected_model_name", "").lower()

    if "light v3" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v3_two_stage_finetuned_from_wlasl1000_train_norm_stats.npz")

    if "light v2" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v2_finetuned_from_wlasl1000_train_norm_stats.npz")

    candidates.extend(sorted(model_dir.glob("*norm_stats*.npz")))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError("Could not find WLASL2000 norm stats .npz file.")

NORM_STATS_FILE = find_norm_stats_file()
stats = np.load(NORM_STATS_FILE)
train_mean = stats["mean"].astype(np.float32)
train_std = stats["std"].astype(np.float32)

print("Norm stats:", NORM_STATS_FILE)

# 5. Define and load model

In [ ]:
class BiGRUAttentionDeploy(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        bi_hidden = hidden_size * 2
        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)
        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)

checkpoint = torch.load(MODEL_PATH, map_location=device)

hidden_size = int(checkpoint.get("hidden_size", 320))
num_layers = int(checkpoint.get("num_layers", 2))
dropout = float(checkpoint.get("dropout", 0.35))

model = BiGRUAttentionDeploy(
    input_size=INPUT_SIZE,
    hidden_size=hidden_size,
    num_classes=NUM_CLASSES,
    num_layers=num_layers,
    dropout=dropout
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded model:", config["selected_model_name"])

# 6. MediaPipe keypoint extraction

In [ ]:
mp_holistic = mp.solutions.holistic

LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4

def extract_landmarks_from_results(results):
    left = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten() if results.left_hand_landmarks else np.zeros(LEFT_HAND_SIZE, dtype=np.float32)
    right = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten() if results.right_hand_landmarks else np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten() if results.pose_landmarks else np.zeros(POSE_SIZE, dtype=np.float32)
    return np.concatenate([left, right, pose]).astype(np.float32)

def process_frame_to_keypoints(frame_bgr, holistic):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = holistic.process(rgb)
    return extract_landmarks_from_results(results)

# 7. Prediction helpers

In [ ]:
def resample_sequence(sequence, target_length=60):
    sequence = np.asarray(sequence, dtype=np.float32)

    if len(sequence) == target_length:
        return sequence

    if len(sequence) == 0:
        return np.zeros((target_length, BASE_FEATURE_SIZE), dtype=np.float32)

    old_x = np.linspace(0, 1, len(sequence))
    new_x = np.linspace(0, 1, target_length)

    resampled = []
    for feature_idx in range(sequence.shape[1]):
        resampled.append(np.interp(new_x, old_x, sequence[:, feature_idx]))

    return np.stack(resampled, axis=1).astype(np.float32)

def prepare_model_input(keypoint_sequence):
    keypoint_sequence = np.asarray(keypoint_sequence, dtype=np.float32)

    normalised = (keypoint_sequence - train_mean.reshape(1, -1)) / (train_std.reshape(1, -1) + 1e-6)

    velocity = np.zeros_like(normalised, dtype=np.float32)
    velocity[1:] = normalised[1:] - normalised[:-1]

    features = np.concatenate([normalised, velocity], axis=1).astype(np.float32)

    return torch.tensor(features, dtype=torch.float32).unsqueeze(0)

def predict_keypoint_sequence(sequence, top_k=5):
    x = prepare_model_input(sequence).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()

    top_ids = np.argsort(probs)[-top_k:][::-1]

    top_predictions = []
    for label_id in top_ids:
        top_predictions.append({
            "label_id": int(label_id),
            "gloss": id_to_gloss.get(int(label_id), str(label_id)),
            "probability": float(probs[label_id])
        })

    return top_predictions

def hand_activity_score(sequence):
    sequence = np.asarray(sequence, dtype=np.float32)

    left_hand = sequence[:, :63]
    right_hand = sequence[:, 63:126]
    hands = np.concatenate([left_hand, right_hand], axis=1)

    non_zero_ratio = np.mean(np.abs(hands) > 1e-6)
    movement = np.mean(np.abs(hands[1:] - hands[:-1]))

    active = non_zero_ratio > 0.05 and movement > 0.002

    return {
        "non_zero_ratio": float(non_zero_ratio),
        "movement": float(movement),
        "active": bool(active)
    }

# 8. Multi-window prediction and event-level Top-5 merge

In [ ]:
FAST_WINDOWS = [30, 45, 60]
TOP_K = 5

def multi_window_predict(keypoint_buffer, top_k=5):
    keypoint_list = list(keypoint_buffer)
    predictions = []

    for window_size in FAST_WINDOWS:
        if len(keypoint_list) < window_size:
            continue

        window = np.array(keypoint_list[-window_size:], dtype=np.float32)
        window = resample_sequence(window, target_length=SEQUENCE_LENGTH)

        top_predictions = predict_keypoint_sequence(window, top_k=top_k)

        predictions.append({
            "window_size": window_size,
            "top_k": top_predictions
        })

    return predictions

def merge_event_predictions(event_predictions, top_k=5):
    scores = defaultdict(float)
    counts = defaultdict(int)
    top1_counts = defaultdict(int)

    for pred_pack in event_predictions:
        for rank, item in enumerate(pred_pack["top_k"]):
            gloss = item["gloss"]
            prob = float(item["probability"])
            rank_bonus = max(0.0, (top_k - rank) / top_k) * 0.03

            scores[gloss] += prob + rank_bonus
            counts[gloss] += 1

            if rank == 0:
                top1_counts[gloss] += 1

    for gloss in list(scores.keys()):
        scores[gloss] += 0.08 * top1_counts[gloss]
        scores[gloss] += 0.02 * counts[gloss]

    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    if not sorted_items:
        return []

    max_score = max(score for _, score in sorted_items)

    merged = []
    for gloss, score in sorted_items:
        merged.append({
            "gloss": gloss,
            "probability": float(score / (max_score + 1e-6)),
            "raw_score": float(score),
            "appearances": int(counts[gloss]),
            "top1_count": int(top1_counts[gloss])
        })

    return merged

# 9. Select screen region

In [ ]:
def capture_full_monitor(monitor_index=1):
    with mss.mss() as sct:
        monitor = sct.monitors[monitor_index]
        screenshot = np.array(sct.grab(monitor))

    frame_bgr = cv2.cvtColor(screenshot, cv2.COLOR_BGRA2BGR)
    return frame_bgr, monitor

def select_screen_region_with_mouse(monitor_index=1):
    full_frame, monitor = capture_full_monitor(monitor_index)

    window_name = "Select Screen Region - ENTER/SPACE confirm, C cancel"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    roi = cv2.selectROI(
        window_name,
        full_frame,
        showCrosshair=True,
        fromCenter=False
    )

    cv2.destroyWindow(window_name)

    x, y, w, h = roi

    if w == 0 or h == 0:
        raise ValueError("No region selected.")

    return {
        "left": int(monitor["left"] + x),
        "top": int(monitor["top"] + y),
        "width": int(w),
        "height": int(h)
    }

def capture_screen_region(region):
    with mss.mss() as sct:
        screenshot = np.array(sct.grab(region))
    return cv2.cvtColor(screenshot, cv2.COLOR_BGRA2BGR)

with mss.mss() as sct:
    for i, monitor in enumerate(sct.monitors):
        print(f"Monitor {i}: {monitor}")

MONITOR_INDEX = 1
SCREEN_REGION = select_screen_region_with_mouse(monitor_index=MONITOR_INDEX)

print("Selected region:")
print(SCREEN_REGION)

preview = capture_screen_region(SCREEN_REGION)
cv2.imshow("Selected region preview - press any key", preview)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 10. Event-based sentence settings

In [ ]:
PREDICT_EVERY_N_FRAMES = 5

SIGN_END_PAUSE_SECONDS = 0.60
SENTENCE_END_PAUSE_SECONDS = 1.80

MIN_EVENT_DURATION_SECONDS = 0.35
MAX_EVENT_DURATION_SECONDS = 2.50

MIN_EVENT_PREDICTIONS = 2
RUN_SECONDS = 45

print("PREDICT_EVERY_N_FRAMES:", PREDICT_EVERY_N_FRAMES)
print("FAST_WINDOWS:", FAST_WINDOWS)
print("SIGN_END_PAUSE_SECONDS:", SIGN_END_PAUSE_SECONDS)
print("SENTENCE_END_PAUSE_SECONDS:", SENTENCE_END_PAUSE_SECONDS)

# 11. Run event-based sentence demo

In [ ]:
WINDOW_NAME = "Be My Ear - Event-Based Sentence Demo"

keypoint_buffer = deque(maxlen=SEQUENCE_LENGTH)

state = "WAITING"
current_event_predictions = []
current_event_start_time = None
last_active_time = None

sentence_events = []
completed_sentences = []

event_id = 0
frame_counter = 0
start_time = time.time()

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, SCREEN_REGION["width"], SCREEN_REGION["height"])

def commit_current_event(now):
    global current_event_predictions, current_event_start_time, event_id, sentence_events

    if current_event_start_time is None:
        current_event_predictions = []
        return None

    duration = now - current_event_start_time

    if duration < MIN_EVENT_DURATION_SECONDS:
        current_event_predictions = []
        return None

    if len(current_event_predictions) < MIN_EVENT_PREDICTIONS:
        current_event_predictions = []
        return None

    merged_top_k = merge_event_predictions(current_event_predictions, top_k=TOP_K)

    if len(merged_top_k) == 0:
        current_event_predictions = []
        return None

    event_id += 1

    event = {
        "event_id": event_id,
        "time": round(current_event_start_time - start_time, 2),
        "duration": round(duration, 2),
        "status": "candidate",
        "top_k": merged_top_k
    }

    sentence_events.append(event)
    current_event_predictions = []

    return event

def commit_sentence_if_ready():
    global sentence_events, completed_sentences

    if len(sentence_events) < 2:
        return None

    result = candidate_sentence_mode(sentence_events[-8:], max_candidates_per_event=3)

    sentence_record = {
        "sentence_id": len(completed_sentences) + 1,
        "sentence": result["sentence"],
        "confidence": result["confidence"],
        "selected_glosses": result["selected_glosses"],
        "events": sentence_events.copy(),
        "details": result
    }

    completed_sentences.append(sentence_record)
    sentence_events = []

    return sentence_record

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while time.time() - start_time < RUN_SECONDS:
        now = time.time()

        frame = capture_screen_region(SCREEN_REGION)

        keypoints = process_frame_to_keypoints(frame, holistic)
        keypoint_buffer.append(keypoints)
        frame_counter += 1

        display_frame = frame.copy()

        if len(keypoint_buffer) >= min(FAST_WINDOWS) and frame_counter % PREDICT_EVERY_N_FRAMES == 0:
            activity_sequence = np.array(list(keypoint_buffer)[-min(len(keypoint_buffer), SEQUENCE_LENGTH):], dtype=np.float32)
            activity = hand_activity_score(activity_sequence)

            if activity["active"]:
                last_active_time = now

                if state == "WAITING":
                    state = "SIGNING"
                    current_event_start_time = now
                    current_event_predictions = []

                multi_preds = multi_window_predict(keypoint_buffer, top_k=TOP_K)

                for pred_pack in multi_preds:
                    current_event_predictions.append(pred_pack)

                if current_event_start_time is not None and now - current_event_start_time >= MAX_EVENT_DURATION_SECONDS:
                    commit_current_event(now)
                    current_event_start_time = now
                    current_event_predictions = []

            else:
                if state == "SIGNING" and last_active_time is not None:
                    inactive_time = now - last_active_time

                    if inactive_time >= SIGN_END_PAUSE_SECONDS:
                        commit_current_event(now)
                        state = "WAITING"
                        current_event_start_time = None

                if last_active_time is not None and now - last_active_time >= SENTENCE_END_PAUSE_SECONDS:
                    if len(sentence_events) >= 2:
                        sentence = commit_sentence_if_ready()
                        if sentence:
                            print("Committed sentence:", sentence["sentence"])

        if len(sentence_events) >= 2:
            live_result = candidate_sentence_mode(sentence_events[-8:], max_candidates_per_event=3)
            live_sentence = live_result["sentence"]
            live_confidence = live_result["confidence"]
        else:
            live_sentence = "Waiting for more sign events..."
            live_confidence = "-"

        lines = [
            f"State: {state}",
            f"Sentence events: {len(sentence_events)} | Completed: {len(completed_sentences)}",
            f"Current sentence: {live_sentence}",
            f"Confidence: {live_confidence}",
            "Q=quit"
        ]

        overlay = display_frame.copy()
        h, w = display_frame.shape[:2]
        cv2.rectangle(overlay, (10, 10), (w - 10, 155), (0, 0, 0), -1)
        display_frame = cv2.addWeighted(overlay, 0.55, display_frame, 0.45, 0)

        for i, line in enumerate(lines):
            cv2.putText(display_frame, line, (20, 40 + i * 28), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 255, 0), 1, cv2.LINE_AA)

        cv2.imshow(WINDOW_NAME, display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

cv2.destroyAllWindows()

now = time.time()
if len(current_event_predictions) >= MIN_EVENT_PREDICTIONS:
    commit_current_event(now)

if len(sentence_events) >= 2:
    commit_sentence_if_ready()

print("Completed sentences:")
for s in completed_sentences:
    print(f"{s['sentence_id']}. {s['sentence']} ({s['confidence']})")

# 12. View events and completed sentences

In [ ]:
event_rows = []

for sentence in completed_sentences:
    for event in sentence["events"]:
        event_rows.append({
            "sentence_id": sentence["sentence_id"],
            "event_id": event["event_id"],
            "time": event["time"],
            "duration": event["duration"],
            "top1": event["top_k"][0]["gloss"],
            "top5": ", ".join([x["gloss"] for x in event["top_k"]])
        })

if event_rows:
    display(pd.DataFrame(event_rows))
else:
    print("No events committed.")

sentence_rows = [
    {
        "sentence_id": s["sentence_id"],
        "sentence": s["sentence"],
        "confidence": s["confidence"],
        "selected_glosses": " → ".join(s["selected_glosses"])
    }
    for s in completed_sentences
]

if sentence_rows:
    display(pd.DataFrame(sentence_rows))
else:
    print("No completed sentences.")

# 13. Save demo output

In [ ]:
REPORT_DIR = PROJECT_ROOT / "reports" / "event_sentence_demo"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

output_file = REPORT_DIR / f"event_based_sentence_demo_{time.strftime('%Y%m%d_%H%M%S')}.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump({
        "screen_region": SCREEN_REGION,
        "settings": {
            "FAST_WINDOWS": FAST_WINDOWS,
            "PREDICT_EVERY_N_FRAMES": PREDICT_EVERY_N_FRAMES,
            "SIGN_END_PAUSE_SECONDS": SIGN_END_PAUSE_SECONDS,
            "SENTENCE_END_PAUSE_SECONDS": SENTENCE_END_PAUSE_SECONDS,
            "MIN_EVENT_DURATION_SECONDS": MIN_EVENT_DURATION_SECONDS,
            "MAX_EVENT_DURATION_SECONDS": MAX_EVENT_DURATION_SECONDS
        },
        "completed_sentences": completed_sentences
    }, f, indent=4)

print("Saved:", output_file)